# ScreamingFace · quickstart

Combine three models into one Fusion, evaluate it on five real GPQA Diamond questions, and compare
the Fusion with its strongest member.

This is the shortest supported path: **configure → compose → evaluate → compare**. The live cell is
disabled initially because it makes provider-backed calls. Enabling it uses real model responses;
disabled mode does not create a report or substitute an offline result.

## Before you run it

Start the local development stack from the repository root:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

The selected model provider credentials must already be available to the stack's AI Gateway.
GPQA is fetched through this notebook's Hugging Face session, so accept its dataset terms and
authenticate this Python environment when required:

```bash
huggingface-cli login
```

The five-case example makes 15 model calls: three Fusion members for each question. Majority vote,
answer-key grading, and the final comparison make no additional provider calls.

## 1 · Configure

In [ ]:
import os

import screamingface as sf

ENGINE_URL = os.environ.get("SCREAMINGFACE_ENGINE_URL", "http://127.0.0.1:4404")
sf.config(engine=ENGINE_URL)

Configuration selects the ScreamingFace engine used for model-backed work. The
localhost value is also the temporary SDK default, while `sf.config(...)` makes it easy to select
another deployment later.

## 2 · Compose

In [ ]:
fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini/2.5",
        "claude/sonnet-4.6",
    ],
    prompt="Return only the answer letter: A, B, C, or D.",
    reducer=sf.reducers.MajorityVote(),
)

fusion

Each member answers the same multiple-choice question. `MajorityVote` selects the
most common exact answer and breaks a tie by stable member order. Fusion construction is local and
does not call a model.

## 3 · Evaluate

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    report = fusion.evaluate("gpqa@1", first=5)
else:
    report = None

report or "Set RUN_LIVE = True when the engine and provider access are ready."

`evaluate(...)` loads the pinned GPQA Diamond definition through this process,
executes the three-member Fusion for the first five canonical cases, checks the answers against the
sealed answer key, and returns one paired comparison. Missing work remains an explicit failure; it
is never silently scored as zero.

## 4 · Compare

In [ ]:
comparison = (
    {
        "score": report.score,
        "baseline": report.baseline,
        "gain": report.gain,
    }
    if report is not None
    else "Run the live evaluation to produce comparison values."
)

comparison

Read `gain` first:

- `score` is the Fusion's accuracy across the successfully paired cases;
- `baseline` is the best individual member's accuracy on those same cases; and
- `gain` is `score - baseline`.

A positive gain means the combination outperformed every member on the evaluated cases. A strong
score with zero gain means the Fusion matched, but did not improve on, its strongest member.

## Recap

```python
sf.config(engine=ENGINE_URL)
fusion = sf.Fusion(..., reducer=sf.reducers.MajorityVote())
report = fusion.evaluate("gpqa@1", first=5)
report.score, report.baseline, report.gain
```

That is the core ScreamingFace workflow. Continue to the engine-profile walkthrough for discovery
and configuration details, or the DRACO walkthrough for model synthesis and rubric judging.